In [ ]:
OUTPUT_PATH = "fig/"

# Logistic loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax, expit
import warnings
import os

def softmax_fn(a):
    return softmax(a)

def loss_fn(W, a, beta):
    """Logistic loss: log(1 + exp(-beta^T * W * sigma(a)))"""
    sigma_a = softmax_fn(a)
    z = np.dot(beta, W @ sigma_a)
    # Use logaddexp for numerical stability: log(1 + exp(-z)) = log(exp(0) + exp(-z))
    return np.logaddexp(0, -z)

def grad_W(W, a, beta):
    """Gradient of L w.r.t. W"""
    sigma_a = softmax_fn(a)
    z = np.dot(beta, W @ sigma_a)
    s = expit(-z) # sigmoid(-z)
    
    # grad_W = -s * beta * sigma(a)^T
    grad = -s * np.outer(beta, sigma_a)
    return grad

def grad_a(W, a, beta):
    """Gradient of L w.r.t. a"""
    sigma_a = softmax_fn(a)
    z = np.dot(beta, W @ sigma_a)
    s = expit(-z) # sigmoid(-z)
    
    # v = -s * W^T * beta
    v = -s * (W.T @ beta)
    
    # Jacobian product: (diag(sigma) - sigma*sigma^T) * v
    grad = sigma_a * v - sigma_a * np.dot(sigma_a, v)
    return grad

def run_experiment(dim=8, learning_rate=0.01, num_iterations=5000, seed=123, num_samples=1000):
    np.random.seed(seed)
    
    # Initialize parameters
    W = 0.1 * np.random.randn(dim, dim)
    a = 0 * np.random.randn(dim)
    beta = np.random.randn(dim)
    beta = beta / np.linalg.norm(beta) # Normalize beta
    
    print(f"Experiment Config: dim={dim}, lr={learning_rate}, iterations={num_iterations}, num_samples={num_samples}")
    print(f"Initial Loss: {loss_fn(W, a, beta):.6f}")

    # Generate logarithmic sample indices
    # We want indices from 0 to num_iterations-1
    # logspace(0, log10(num_iterations), ...) gives values from 1 to num_iterations
    # Subtract 1 to get 0 to num_iterations-1
    sample_indices = np.unique(np.logspace(0, np.log10(num_iterations), num=num_samples).astype(int)) - 1
    # Filter to ensure valid range [0, num_iterations-1]
    sample_indices = sample_indices[(sample_indices >= 0) & (sample_indices < num_iterations)]
    sample_set = set(sample_indices)
    # Always include first and last
    sample_set.add(0)
    sample_set.add(num_iterations - 1)

    # History storage
    history = {
        'loss': [],
        'W': [],
        'a': [],
        'u': [],
        'grad_W_norm': [],
        'grad_a_norm': [],
        'alignment': [],
        'predictor_norm': [],
        'iterations': []
    }

    # Gradient Descent Loop
    for i in range(num_iterations):
        gW = grad_W(W, a, beta)
        ga = grad_a(W, a, beta)
        
        W -= learning_rate * gW
        a -= learning_rate * ga
        
        # Record history
        if i in sample_set:
            # Compute metrics
            loss = loss_fn(W, a, beta)
            predictor = W @ softmax_fn(a)
            pred_norm = np.linalg.norm(predictor)
            alignment = np.dot(beta, predictor) / (pred_norm + 1e-8)
            u = W.T @ beta
            
            history['loss'].append(loss)
            history['W'].append(W.copy())
            history['a'].append(a.copy())
            history['u'].append(u.copy())
            history['grad_W_norm'].append(np.linalg.norm(gW))
            history['grad_a_norm'].append(np.linalg.norm(ga))
            history['alignment'].append(alignment)
            history['predictor_norm'].append(pred_norm)
            history['iterations'].append(i)
        
        if (i + 1) % (num_iterations // 5) == 0:
            # Recompute for printing if not just computed
            if i not in sample_set:
                loss = loss_fn(W, a, beta)
                predictor = W @ softmax_fn(a)
                pred_norm = np.linalg.norm(predictor)
                alignment = np.dot(beta, predictor) / (pred_norm + 1e-8)
            print(f"Iter {i+1:4d} | Loss: {loss:.6f} | Align: {alignment:.4f} | Norm: {pred_norm:.4f}")

    print(f"Final Loss: {history['loss'][-1]:.6f}")
    return history, W, a, beta

In [ ]:
history, W_final, a_final, beta = run_experiment(dim=8, learning_rate=0.1, num_iterations=int(1e6), num_samples=2000)

# -------------------------
# ICML-style settings
# -------------------------
plt.rcParams.update({
    "text.usetex": False,  # set to true for better captions, but then you need to install latex.
    "font.family": "serif",
    "font.serif": ["Times", "Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# -------------------------
# Data
# -------------------------
a_hist = np.asarray(history["a"])
iterations = np.asarray(history["iterations"])
dim = a_hist.shape[1]

# -------------------------
# Colorscheme
# -------------------------
base = sns.color_palette("hls", max(8, dim))
priority = [3, 5, 1, 7, 0, 2, 4, 6] + list(range(8, max(8, dim)))
colors = [base[i] for i in priority[:dim]]

# -------------------------
# Figure size: ICML two-column width
# -------------------------
TWO_COL_WIDTH_IN = 6.75  # good default for ICML two-column figures
HEIGHT_IN = 2         # adjust if legends get tall
fig, axes = plt.subplots(1, 2, figsize=(TWO_COL_WIDTH_IN, HEIGHT_IN))

# -------------------------
# Left: softmax evolution (log-log)
# -------------------------
softmax_hist = np.asarray([softmax_fn(a) for a in a_hist])  # (T, dim)
for d, c in enumerate(colors):
    axes[0].loglog(
        iterations,
        softmax_hist[:, d],
        label=rf"$\sigma(a)_{{{d+1}}}$",
        color=c,
        alpha=0.9,
        lw=1.2,
    )
axes[0].set_xlabel(r"Iteration $t$")
axes[0].set_ylabel(r"Value")
axes[0].set_title(r"Evolution of $\sigma(a)$")
axes[0].legend(frameon=False, handlelength=1.6, borderaxespad=0.2, loc='lower left', ncol=2)

# -------------------------
# Right: u evolution (semi-log x)
# -------------------------
if "u" in history:
    u_hist = np.asarray(history["u"])  # (T, dim)
    for d, c in enumerate(colors):
        axes[1].semilogx(
            iterations,
            u_hist[:, d],
            label=rf"$u_{{{d+1}}}$",
            color=c,
            alpha=0.9,
            lw=1.2,
        )
    axes[1].set_ylabel(r"Value")
    axes[1].set_title(r"Evolution of $u$")
    axes[1].legend(frameon=False, handlelength=1.6, borderaxespad=0.2, loc='upper left', ncol=2)
else:
    axes[1].axis("off")

axes[1].set_xlabel(r"Iteration $t$")



# Small cosmetics consistent with the paper look
for ax in axes:
    ax.tick_params(direction="out", length=2.5, width=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    # Remove minor ticks, keep only major ticks
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())

fig.tight_layout(pad=0.2, w_pad=0.6)
fig.savefig(os.path.join(OUTPUT_PATH, "softmax_and_u_evolution_icml_2col.pdf"))
plt.show()

# Comparison of different attention types

In [ ]:
class NormalizationModule:
    """Base class for normalization strategies."""
    def forward(self, z):
        raise NotImplementedError
    def backward(self, grad_output, z, output):
        raise NotImplementedError

class L1Normalization(NormalizationModule):
    def forward(self, z):
        self.norm = np.sum(np.abs(z)) + 1e-10
        return np.abs(z) / self.norm

    def backward(self, grad_output, z, alpha):
        # L1 Gradient Dynamics
        baseline = np.sum(grad_output * alpha)
        sign_z = np.sign(z)
        grad_input = (grad_output - baseline) * (sign_z / self.norm)
        return grad_input

class SumNormalization(NormalizationModule):
    def forward(self, z):
        # Sum Normalization: z / sum(z)
        self.norm = np.sum(z) + 1e-10
        return z / self.norm

    def backward(self, grad_output, z, alpha):
        # Sum Normalization Gradient Dynamics
        baseline = np.sum(grad_output * alpha)
        grad_input = (grad_output - baseline) / self.norm
        return grad_input

class L2ActivationNormalization(NormalizationModule):
    def forward(self, z):
        # L2 Activation: z^2 / sum(z^2)
        self.norm = np.sum(z**2) + 1e-10
        return (z**2) / self.norm

    def backward(self, grad_output, z, alpha):
        # L2 Activation Gradient Dynamics
        baseline = np.sum(grad_output * alpha)
        grad_input = (2 * z / self.norm) * (grad_output - baseline)
        return grad_input

class SoftmaxNormalization(NormalizationModule):
    def forward(self, z):
        # Numerical stability shift
        e_x = np.exp(z - np.max(z))
        return e_x / e_x.sum()

    def backward(self, grad_output, z, alpha):
        # Softmax Gradient Dynamics (Replicator)
        baseline = np.sum(grad_output * alpha)
        grad_input = alpha * (grad_output - baseline)
        return grad_input

class SigmoidNormalization(NormalizationModule):
    def forward(self, z):
        # Numerical stability shift
        return expit(z)

    def backward(self, grad_output, z, alpha):
        # Softmax Gradient Dynamics (Replicator)
        baseline = np.sum(grad_output * alpha)
        grad_input = expit(z) * (1 - expit(z)) * (grad_output)
        return grad_input

class ReluNormalization(NormalizationModule):
    def forward(self, z):
        # Numerical stability shift
        self.norm = np.sum(np.maximum(0, z)) + 1e-10
        return np.maximum(0, z) / self.norm

    def backward(self, grad_output, z, alpha):
        # Softmax Gradient Dynamics (Replicator)
        baseline = np.sum(grad_output * alpha)
        grad_input = (grad_output - baseline) * (z > 0) / self.norm
        return grad_input

class BaseSolver:
    def __init__(self, n_features, n_latent, norm_module, learning_rate=0.01, scale_V=1.0, scale_a=1.0):
        self.lr = learning_rate
        self.norm_module = norm_module
        np.random.seed(42)
        self.V = scale_V * np.random.randn(n_features, n_latent)
        self.a = scale_a * (np.ones(n_latent) + np.random.randn(n_latent) * 0.1)
        self.history = {'alpha': [], 'loss': [], 'a': []}

    def step(self, target_beta):
        raise NotImplementedError

class SquareLossSolver(BaseSolver):
    def step(self, target_beta):
        # Forward
        alpha = self.norm_module.forward(self.a)
        prediction = self.V @ alpha
        residual = prediction - target_beta
        loss = 0.5 * np.sum(residual**2)
        
        # Backward
        grad_prediction = residual
        grad_V = np.outer(grad_prediction, alpha)
        grad_alpha = self.V.T @ grad_prediction
        grad_a = self.norm_module.backward(grad_alpha, self.a, alpha)
        
        # Update
        self.V -= self.lr * grad_V
        self.a -= self.lr * grad_a
        
        self.history['alpha'].append(alpha.copy())
        self.history['loss'].append(loss)
        self.history['a'].append(self.a.copy())

class LogisticLossSolver(BaseSolver):
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def step(self, target_beta):
        # Forward
        alpha = self.norm_module.forward(self.a)
        reconstruction = self.V @ alpha
        score = np.dot(target_beta, reconstruction)
        
        # Logistic Loss: log(1 + exp(-score))
        if score > 0:
            loss = np.log(1 + np.exp(-score))
        else:
            loss = -score + np.log(1 + np.exp(score))
            
        # Backward
        grad_score = -(1 - self.sigmoid(score))
        grad_V = grad_score * np.outer(target_beta, alpha)
        grad_alpha = grad_score * (target_beta @ self.V)
        grad_a = self.norm_module.backward(grad_alpha, self.a, alpha)
        
        # Update
        self.V -= self.lr * grad_V
        self.a -= self.lr * grad_a
        
        self.history['alpha'].append(alpha.copy())
        self.history['loss'].append(loss)
        self.history['a'].append(self.a.copy())

def run_experiment(solver_class, loss_name, n_iter=2000, lr=0.1, V_scale=1.0):
    N_FEATURES = 10
    N_LATENT = 10
    
    np.random.seed(42)
    target_beta = np.random.randn(N_FEATURES)
    target_beta /= np.linalg.norm(target_beta)
    
    print(f"--- Running {loss_name} Experiment ---")
    
    # Run L1
    print(f"  > Training L1...")
    l1_solver = solver_class(N_FEATURES, N_LATENT, L1Normalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        l1_solver.step(target_beta)

    # Run Sum
    print(f"  > Training Sum...")
    sum_solver = solver_class(N_FEATURES, N_LATENT, SumNormalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        sum_solver.step(target_beta)

    # Run L2 Activation
    print(f"  > Training L2 Activation...")
    l2_solver = solver_class(N_FEATURES, N_LATENT, L2ActivationNormalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        l2_solver.step(target_beta)
        
    # Run Softmax
    print(f"  > Training Softmax...")
    sm_solver = solver_class(N_FEATURES, N_LATENT, SoftmaxNormalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        sm_solver.step(target_beta)

    # Run Relu
    print(f"  > Training Relu...")
    relu_solver = solver_class(N_FEATURES, N_LATENT, ReluNormalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        relu_solver.step(target_beta)

    # Run Sigmoid
    print(f"  > Training Sigmoid...")
    sig_solver = solver_class(N_FEATURES, N_LATENT, SigmoidNormalization(), learning_rate=lr, scale_V=V_scale)
    for _ in range(n_iter):
        sig_solver.step(target_beta)
        
    return l1_solver.history, sum_solver.history, l2_solver.history, sm_solver.history, relu_solver.history, sig_solver.history

def plot_combined_results(l1_hist, sum_hist, l2_hist, sm_hist, relu_hist, sig_hist, title, alpha_decimals=3):
    fig, axes = plt.subplots(6, 2, figsize=(3.25, 7.5))
    # fig.suptitle(rf'\textbf{{{title}}}', fontsize=10)
    
    histories = [
        (l1_hist, "L1"),
        (sum_hist, "Sum"),
        (l2_hist, "L2"),
        (sm_hist, "Softmax"),
        (relu_hist, "Relu"),
        (sig_hist, "Sigmoid")
    ]
    
    # Set column titles once
    axes[0, 0].set_title(r'Weight $\alpha$')
    axes[0, 1].set_title(r'Value $a$')
    
    for row, (hist, name) in enumerate(histories):
        iterations = list(range(len(hist['alpha'])))
        alpha_arr = np.array(hist['alpha'])
        a_arr = np.array(hist['a'])
        dim = alpha_arr.shape[1]
        colors = sns.color_palette("husl", dim)
        
        # Left column: Alphas
        for d in range(dim):
            axes[row, 0].plot(iterations, alpha_arr[:, d], color=colors[d], alpha=0.8, lw=1)

        if row == 5:
            axes[row, 0].set_xlabel(r'Iteration $t$')
        axes[row, 0].set_ylabel(rf'\textbf{{{name}}}')
        axes[row, 0].set_xscale('log')
        axes[row, 0].yaxis.set_major_formatter(plt.FormatStrFormatter(f'%.{alpha_decimals}f'))
        
        # Right column: Pre-activations
        for d in range(dim):
            axes[row, 1].plot(iterations, a_arr[:, d], color=colors[d], alpha=0.8, lw=1)

        if row == 5:
            axes[row, 1].set_xlabel(r'Iteration $t$')
        axes[row, 1].set_xscale('log')
    
    plt.tight_layout()

In [ ]:
# 1. Square Loss Experiment
print("\n=== SQUARE LOSS EXPERIMENT ===")
ret_sq = run_experiment(SquareLossSolver, "Square Loss", n_iter=5000, lr=0.1, V_scale=0.1)
plot_combined_results(*ret_sq, "Square Loss Comparison", alpha_decimals=3)
plt.savefig(os.path.join(OUTPUT_PATH, "square_loss_comparison.pdf"))
plt.show()

# 2. Logistic Loss Experiment
print("\n=== LOGISTIC LOSS EXPERIMENT ===")
ret_log = run_experiment(LogisticLossSolver, "Logistic Loss", n_iter=100000, lr=0.5, V_scale=0.1)
plot_combined_results(*ret_log, "Logistic Loss Comparison", alpha_decimals=1)
plt.savefig(os.path.join(OUTPUT_PATH, "log_loss_comparison.pdf"))
plt.show()

# Tied value-softmax model

In [ ]:
class TiedUSolver(BaseSolver):
    def __init__(self, n_features, n_latent, norm_module, learning_rate=0.01, scale_V=1.0, scale_a=1.0):
        # Initialize base (creates self.V, self.a) - note BaseSolver creates V and a
        # We will use self.V as our 'U' matrix
        super().__init__(n_features, n_latent, norm_module, learning_rate, scale_V, scale_a)
        
        # Override V (U) initialization to Identity if desired, or keep random
        # User requested: "U,V are matrices initialized on indentity" in previous prompt, 
        # and "U initialized on identity" implicitly here.
        # Assuming n_features == n_latent for Identity to make sense, or rectangular identity.
        
        self.U = np.eye(n_features, n_latent) * scale_V
        # self.a is already initialized by BaseSolver
        
        self.history = {'alpha': [], 'loss': [], 'a': [], 'U': []}

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def step(self, target_beta):
        # Forward
        # z = U a
        z = self.U @ self.a
        
        # alpha = softmax(z)
        # Note: BaseSolver uses self.norm_module on self.a directly. 
        # Here we apply it on z.
        alpha = self.norm_module.forward(z)
        
        # beta = U alpha
        beta = self.U @ alpha
        
        # Loss: log(1 + exp(-beta_*^T beta))
        score = np.dot(target_beta, beta)
        
        if score > 0:
            loss = np.log(1 + np.exp(-score))
        else:
            loss = -score + np.log(1 + np.exp(score))
            
        # Backward
        # dL/d(beta)
        grad_score = -(1 - self.sigmoid(score))
        grad_beta = grad_score * target_beta
        
        # beta = U alpha
        # grad_U_1 = grad_beta @ alpha^T
        grad_U_from_beta = np.outer(grad_beta, alpha)
        
        # grad_alpha = U^T @ grad_beta
        grad_alpha = self.U.T @ grad_beta
        
        # alpha = softmax(z)
        # grad_z = J_softmax^T @ grad_alpha
        # norm_module.backward computes J^T @ grad_output. 
        # CAUTION: norm_module.backward usually takes (grad_output, input_to_softmax, output_of_softmax)
        # Here input is z, output is alpha.
        grad_z = self.norm_module.backward(grad_alpha, z, alpha)
        
        # z = U a
        # grad_U_2 = grad_z @ a^T
        grad_U_from_z = np.outer(grad_z, self.a)
        
        # grad_a = U^T @ grad_z
        grad_a = self.U.T @ grad_z
        
        # Total grad U
        grad_U = grad_U_from_beta + grad_U_from_z
        
        # Update
        self.U -= self.lr * grad_U
        self.a -= self.lr * grad_a
        
        self.history['alpha'].append(alpha.copy()) # Softmax scores
        self.history['loss'].append(loss)
        self.history['a'].append(self.a.copy()) # Parameter a
        self.history['U'].append(self.U.copy())

# Run the Tied U Experiment
print("\n=== TIED U SOFTMAX (LOGISTIC LOSS) EXPERIMENT ===")

Ds = [10, 20, 50, 100]
n_rows = len(Ds)
fig, axes = plt.subplots(n_rows, 3, figsize=(6.75, 1.5 * n_rows))

for row, D in enumerate(Ds):
    N_FEATURES = D
    N_LATENT = D
    n_iter = 10000
    lr = 0.05

    np.random.seed(42)
    target_beta = np.random.randn(N_FEATURES)
    target_beta /= np.linalg.norm(target_beta)

    print(f"Training TiedUSolver for D={D}...")
    # Using SoftmaxNormalization
    tied_u_solver = TiedUSolver(N_FEATURES, N_LATENT, SoftmaxNormalization(), learning_rate=lr, scale_V=1.0) 

    for _ in range(n_iter):
        tied_u_solver.step(target_beta)

    iterations = list(range(len(tied_u_solver.history['loss'])))

    # 1. Loss
    axes[row, 0].plot(iterations, tied_u_solver.history['loss'], lw=1, color=sns.color_palette("deep")[0])
    axes[row, 0].set_xlabel(r'Iteration $t$')
    axes[row, 0].set_ylabel(r'Loss $L(U, a)$')
    axes[row, 0].set_title(rf'\textbf{{Loss}} ($D = {D}$)')
    axes[row, 0].set_yscale('log')
    axes[row, 0].set_xscale('log')

    # 2. Softmax Scores (alpha)
    alpha_hist = np.array(tied_u_solver.history['alpha'])
    dim = alpha_hist.shape[1]
    colors = sns.color_palette("husl", dim)
    for d in range(dim):
        axes[row, 1].plot(iterations, alpha_hist[:, d], color=colors[d], alpha=0.8, lw=1)
    axes[row, 1].set_xlabel(r'Iteration $t$')
    axes[row, 1].set_ylabel(r'Probability')
    axes[row, 1].set_title(rf'$\sigma(Ua)$ ($D = {D}$)')
    axes[row, 1].set_xscale('log')

    u_col_norms = []
    for alpha, U in zip(tied_u_solver.history['alpha'], tied_u_solver.history['U']):
        a_max_idx = np.argmax(alpha)
        u_col_norms.append(np.linalg.norm(U[:, a_max_idx]))

    # 3. Norm of U
    # u_norms = [np.linalg.norm(u) for u in tied_u_solver.history['U']]
    axes[row, 2].plot(iterations, u_col_norms, lw=1, color=sns.color_palette("deep")[2])
    axes[row, 2].set_xlabel(r'Iteration $t$')
    axes[row, 2].set_ylabel(r'$\|U^{(m)}\|_F$')
    axes[row, 2].set_title(rf'$\|U^{{(m)}}\|_F$ ($D = {D}$)')
    axes[row, 2].set_xscale('log')

plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_PATH, "massive_activations_2.pdf"))

# Regression with different condition numbers

In [ ]:
def softmax_fn(a):
    """Compute softmax of a vector."""
    return softmax(a)

def loss_fn(W, a, X, beta):
    """Compute loss L(W,a) = ||beta - X*W*softmax(a)||^2"""
    sigma_a = softmax_fn(a)
    pred = X @ (W @ sigma_a)
    return np.sum((beta - pred) ** 2)

# Test the loss function
np.random.seed(1729)
dim = 8
n_samples = 20
W_test = np.random.randn(dim, dim)
a_test = np.random.randn(dim)
X_test = np.random.randn(n_samples, dim)
beta_test = np.random.randn(n_samples)
loss_test = loss_fn(W_test, a_test, X_test, beta_test)
print(f"Loss function test: L = {loss_test:.6f}")

def grad_W(W, a, X, beta):
    """Gradient of L w.r.t. W: -2 * X^T * residual * sigma(a)^T"""
    sigma_a = softmax_fn(a)
    residual = beta - X @ (W @ sigma_a)
    # grad = -2 * (X.T @ residual) @ sigma_a.T
    grad = -2 * np.outer(X.T @ residual, sigma_a)
    return grad

def grad_a(W, a, X, beta):
    """Gradient of L w.r.t. a: Jacobian^T * (-2 * W^T * X^T * residual)"""
    sigma_a = softmax_fn(a)
    residual = beta - X @ (W @ sigma_a)
    v = -2 * W.T @ (X.T @ residual)
    
    # Jacobian of softmax is diag(sigma) - sigma * sigma^T
    # We need J^T * v = (diag(sigma) - sigma * sigma^T) * v
    # = sigma * v - sigma * (sigma^T * v)
    grad = sigma_a * v - sigma_a * np.dot(sigma_a, v)
    return grad

def run_experiment(dim=8, n_samples=20, learning_rate=0.01, num_iterations=1000, seed=123, condition_number=100, num_history_samples=1000):
    np.random.seed(seed)
    
    # Initialize parameters
    W = 0.00001*np.random.randn(dim, dim)
    a = 0.001 * np.random.randn(dim)
    
    # Create ill-conditioned X
    # 1. Generate random matrix
    X_raw = np.random.randn(n_samples, dim)
    # 2. SVD
    U, S, Vt = np.linalg.svd(X_raw, full_matrices=False)
    # 3. Modify singular values to be ill-conditioned
    # Linearly spaced singular values from 1 to 1/condition_number
    S_new = np.linspace(1, 1/condition_number, min(n_samples, dim))
    # 4. Reconstruct X
    X = U @ np.diag(S_new) @ Vt
    
    beta = np.random.randn(n_samples)
    
    print(f"Experiment Config: dim={dim}, n_samples={n_samples}, lr={learning_rate}, iterations={num_iterations}")
    print(f"X Condition Number: {np.linalg.cond(X):.2f}")
    print(f"Initial Loss: {loss_fn(W, a, X, beta):.6f}")

    # Determine sampling indices (logarithmic)
    if num_iterations > 1:
        sample_indices = np.unique(np.logspace(0, np.log10(num_iterations-1), num_history_samples).astype(int))
        sample_indices = np.insert(sample_indices, 0, 0)
    else:
        sample_indices = np.array([0])
    sample_indices_set = set(sample_indices)

    # History storage
    history = {
        'loss': [],
        'W': [],
        'a': [],
        'grad_W_norm': [],
        'grad_a_norm': [],
        'iterations': [],
        'W_svd': []
    }

    # Gradient Descent Loop
    for i in range(num_iterations):
        gW = grad_W(W, a, X, beta)
        ga = grad_a(W, a, X, beta)
        
        W -= learning_rate * gW
        a -= learning_rate * ga
        
        # Record history at sampled iterations
        if i in sample_indices_set:
            history['loss'].append(loss_fn(W, a, X, beta))
            history['W'].append(W.copy())
            history['a'].append(a.copy())
            history['grad_W_norm'].append(np.linalg.norm(gW))
            history['grad_a_norm'].append(np.linalg.norm(ga))
            history['iterations'].append(i)
            history['W_svd'].append(np.linalg.svd(W, compute_uv=False))
        
        if (i + 1) % (num_iterations // 5) == 0:
            current_loss = history['loss'][-1] if i in sample_indices_set else loss_fn(W, a, X, beta)
            print(f"Iter {i+1:4d} | Loss: {current_loss:.6f} | ||gW||: {np.linalg.norm(gW):.4f} | ||ga||: {np.linalg.norm(ga):.4f}")

    print(f"Final Loss: {loss_fn(W, a, X, beta):.6f}")
    return history, W, a, X, beta

def multiple_experiments(condition_numbers):
    histories = []
    for condition_number in condition_numbers:
        history, W_final, a_final, X, beta = run_experiment(dim=20, n_samples=20, learning_rate=0.001, num_iterations=300000, condition_number=condition_number, num_history_samples=3000)
        histories.append(history)
    return histories

def plot_multiple_results(condition_numbers, histories):
    n_rows = len(condition_numbers)
    fig, axes = plt.subplots(n_rows, 3, figsize=(6.75, 1.5 * n_rows))
    
    for row, (condition_number, history) in enumerate(zip(condition_numbers, histories)):
        iterations = history['iterations']
        
        # 1. Loss
        axes[row, 0].plot(iterations, history['loss'], lw=1, color=sns.color_palette("deep")[0])
        axes[row, 0].set_xlabel(r'Iteration $t$')
        axes[row, 0].set_ylabel(r'Loss $L(W, a)$')
        axes[row, 0].set_title(rf'\textbf{{Loss Trajectory}} ($\kappa = {condition_number}$)')
        axes[row, 0].set_yscale('log')
        # axes[row, 0].set_xscale('log')
        
        # 2. Gradient Norm
        axes[row, 1].plot(iterations, history['grad_W_norm'], label=r'$\|\nabla_W L\|$', lw=1, color=sns.color_palette("husl")[1])
        axes[row, 1].plot(iterations, history['grad_a_norm'], label=r'$\|\nabla_a L\|$', lw=1, color=sns.color_palette("husl")[2])
        axes[row, 1].set_xlabel(r'Iteration $t$')
        axes[row, 1].set_ylabel(r'Gradient Norm')
        axes[row, 1].set_title(rf'\textbf{{Gradient Norms}} ($\kappa = {condition_number}$)')
        axes[row, 1].set_yscale('log')
        # axes[row, 1].set_xscale('log')
        if row == 0:
            axes[row, 1].legend(frameon=False, loc='upper right')
        
        # 3. Parameter a evolution
        a_hist = np.array(history['a'])
        dim = a_hist.shape[1]
        colors = sns.color_palette("husl", dim)
        for d in range(dim):
            axes[row, 2].plot(iterations, a_hist[:, d], label=f'$a_{{{d+1}}}$', color=colors[d], alpha=0.8, lw=1)
        axes[row, 2].set_xlabel(r'Iteration $t$')
        axes[row, 2].set_ylabel(r'Parameter Value')
        axes[row, 2].set_title(rf'\textbf{{Evolution of}} $a$ ($\kappa = {condition_number}$)')
        # axes[row, 2].set_xscale('log')
    
    plt.tight_layout()

def plot_a_evolution_comparison(condition_numbers, histories, selected_kappas=[1, 5]):
    # Find indices for selected kappas
    indices = [i for i, k in enumerate(condition_numbers) if k in selected_kappas]
    
    fig, axes = plt.subplots(1, 2, figsize=(3.375, 1.3))
    
    for ax_idx, idx in enumerate(indices):
        condition_number = condition_numbers[idx]
        history = histories[idx]
        iterations = history['iterations']
        
        # Parameter a evolution
        a_hist = np.array(history['a'])
        dim = a_hist.shape[1]
        colors = sns.color_palette("husl", dim)
        for d in range(dim):
            axes[ax_idx].plot(iterations, a_hist[:, d], label=f'$a_{{{d+1}}}$', color=colors[d], alpha=0.8, lw=0.7)
        # axes[ax_idx].set_xlabel(r'Iteration $t$')
        # if ax_idx == 0:
            #axes[ax_idx].set_ylabel(r'Parameter Value')
        axes[ax_idx].set_title(rf'$\kappa = {condition_number}$')
        # axes[ax_idx].set_xscale('log')
    
    plt.tight_layout()

In [ ]:
cond_numbers = [1, 2, 3, 4, 5]

histories = multiple_experiments(cond_numbers)

condition_numbers = cond_numbers
plot_multiple_results(condition_numbers, histories)
plt.savefig(os.path.join(OUTPUT_PATH, "condition_numbers.pdf"))

plot_a_evolution_comparison(condition_numbers, histories, selected_kappas=[1, 5])
plt.savefig(os.path.join(OUTPUT_PATH, "condition_numbers_small.pdf"))

# KL-divergence

In [ ]:
def softmax_fn(a):
    return softmax(a)
		
def loss_fn(V, a, p_star):
    s_a = softmax(a)
    z = V.T @ s_a
    q = softmax(z)
    # Add small epsilon for numerical stability
    return -np.sum(p_star * np.log(q + 1e-15))

def grad_V(V, a, p_star):
    s_a = softmax(a)
    z = V.T @ s_a
    q = softmax(z)
    
    delta_z = q - p_star
    # Grad_V = s_a * delta_z^T
    return np.outer(s_a, delta_z)

def grad_a(V, a, p_star):
    s_a = softmax(a)
    z = V.T @ s_a
    q = softmax(z)
    
    delta_z = q - p_star
    
    # Backprop to s_a
    # dL/ds_a = V * delta_z
    grad_s_a = V @ delta_z
    
    # Backprop through softmax(a)
    # dL/da = (diag(s) - s s^T) * grad_s_a
    #       = s * grad_s_a - s * (s^T grad_s_a)
    term = np.dot(s_a, grad_s_a)
    grad_a = s_a * grad_s_a - s_a * term
    return grad_a

def run_experiment(d=8, k=4, learning_rate=0.01, num_iterations=1000, seed=123, num_samples=1000):
    np.random.seed(seed)
    
    # Initialize parameters
    V = 0.1 * np.random.randn(d, k)
    a = 0* np.random.randn(d)
    
    # Target distribution p_star
    p_star = np.random.rand(k)
    p_star = softmax(10 * p_star) # Make it somewhat sharp
    
    print(f"Experiment Config: d={d}, k={k}, lr={learning_rate}, iterations={num_iterations}")
    print(f"Initial Loss: {loss_fn(V, a, p_star):.6f}")

    # Logarithmic sampling indices
    if num_iterations > 1:
        sample_indices = np.unique(np.logspace(0, np.log10(num_iterations-1), num_samples).astype(int))
        sample_indices = np.insert(sample_indices, 0, 0)
    else:
        sample_indices = np.array([0])
    sample_indices_set = set(sample_indices)

    # History
    history = {
        'loss': [],
        'grad_V_norm': [],
        'grad_a_norm': [],
        'a': [],
        'iterations': []
    }

    # Training Loop
    for i in range(num_iterations):
        gV = grad_V(V, a, p_star)
        ga = grad_a(V, a, p_star)
        
        V -= learning_rate * gV
        a -= learning_rate * ga
        
        if i in sample_indices_set:
            history['loss'].append(loss_fn(V, a, p_star))
            history['grad_V_norm'].append(np.linalg.norm(gV))
            history['grad_a_norm'].append(np.linalg.norm(ga))
            history['a'].append(a.copy())
            history['iterations'].append(i)
        
        if (i + 1) % (num_iterations // 5) == 0:
             current_loss = history['loss'][-1] if i in sample_indices_set else loss_fn(V, a, p_star)
             print(f"Iter {i+1:4d} | Loss: {current_loss:.6f}")

    print(f"Final Loss: {loss_fn(V, a, p_star):.6f}")
    return history, V, a, p_star

In [ ]:
num_iterations = int(1e7)
history, V_final, a_final, p_star = run_experiment(d=10, k=10, learning_rate=0.1, num_iterations=num_iterations, num_samples=1000)

def plot_results(history):
    # Configure plotting style
    sns.set_theme(style="white", context="paper", font_scale=1.)
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Times", "Times New Roman", "DejaVu Serif"],
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    
    iterations = history['iterations']
    fig, axes = plt.subplots(1, 3, figsize=(6.75, 2))
    
    # 1. Loss
    axes[0].plot(iterations, history['loss'], lw=1, color=sns.color_palette("deep")[0])
    axes[0].set_xscale('log')
    axes[0].set_xlabel(r'Iteration $t$')
    axes[0].set_ylabel(r'Loss')
    axes[0].set_title(r'\textbf{Loss Trajectory}')
    axes[0].set_yscale('log')
    # axes[0].set_yticks([1.25, 1.5, 1.75, 2.0])
    
    # 2. Grads
    axes[1].plot(iterations, history['grad_V_norm'], label=r'$\|\nabla_V L\|$', lw=1, color=sns.color_palette("deep")[1])
    axes[1].plot(iterations, history['grad_a_norm'], label=r'$\|\nabla_a L\|$', lw=1, color=sns.color_palette("deep")[2])
    axes[1].set_xlabel(r'Iteration $t$')
    axes[1].set_ylabel(r'Norm')
    axes[1].set_title(r'\textbf{Gradient Norms}')
    axes[1].set_yscale('log')
    axes[1].set_xscale('log')
    axes[1].legend(frameon=False)
    
    # 3. a evolution
    a_hist = np.array(history['a'])
    d = a_hist.shape[1]
    colors = sns.color_palette("husl", d)
    for i in range(d):
        axes[2].plot(iterations, a_hist[:, i], color=colors[i], alpha=0.8, lw=1)
    axes[2].set_xlabel(r'Iteration $t$')
    axes[2].set_title(r'\textbf{Evolution of} $a$')
    axes[2].set_xscale('log')
    
    plt.tight_layout()

plot_results(history)
plt.savefig(os.path.join(OUTPUT_PATH, "kl.pdf"))